<a href="https://colab.research.google.com/github/snehapadgaonkar/beat2bit/blob/main/notebooks/03_baseline_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Milestone 2: Baseline 1D CNN
This notebook builds and trains the baseline Convolutional Neural Network
using the extracted MIT-BIH dataset.

In [30]:
# Install requirements (uncomment if running in Colab)
!pip install wfdb numpy matplotlib scipy pandas

In [31]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report
import os
import time

### 1. Load the Preprocessed Dataset

In [32]:
print("Loading preprocessed dataset...")
X_train = np.load('/content/drive/MyDrive/data/processed/X_train.npy')
y_train = np.load('/content/drive/MyDrive/data/processed/y_train.npy')
X_test = np.load('/content/drive/MyDrive/data/processed/X_test.npy')
y_test = np.load('/content/drive/MyDrive/data/processed/y_test.npy')

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

Loading preprocessed dataset...
X_train: (51002, 180, 1), y_train: (51002,)
X_test: (50262, 180, 1), y_test: (50262,)


### 2. Build the 1D CNN Architecture
A lightweight architecture strictly designed for Edge device constraints.

In [33]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(180, 1)),
    tf.keras.layers.Conv1D(16, kernel_size=7, activation='relu', padding='same'),
    tf.keras.layers.MaxPooling1D(pool_size=2),
    tf.keras.layers.Conv1D(32, kernel_size=5, activation='relu', padding='same'),
    tf.keras.layers.MaxPooling1D(pool_size=2),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_2 (Conv1D)               │ (None, 180, 16)        │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 90, 16)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 90, 32)         │         2,592 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_3 (MaxPooling1D)  │ (None, 45, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ (None, 32)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,265 (12.75 KB)

 Trainable params: 3,265 (12.75 KB)

 Non-trainable params: 0 (0.00 B)

### 3. Handle Class Imbalance & Train
We compute class weights since Normal beats heavily outnumber Abnormal beats.

In [34]:
neg, pos = np.bincount(y_train)
total = neg + pos
class_weight = {0: (1 / neg)*(total/2.0), 1: (1 / pos)*(total/2.0)}
print(f"Class Weights: {class_weight}")

print("Starting Training...")
start_time = time.time()
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    class_weight=class_weight,
    verbose=1
)
print(f"Training finished in {time.time() - start_time:.2f} seconds.")

Class Weights: {0: np.float64(0.5561104326587578), 1: np.float64(4.9554994170229305)}
Starting Training...
Epoch 1/10
359/359 ━━━━━━━━━━━━━━━━━━━━ 11s 27ms/step - accuracy: 0.7762 - loss: 0.5399 - val_accuracy: 0.7036 - val_loss: 0.5337
Epoch 2/10
359/359 ━━━━━━━━━━━━━━━━━━━━ 12s 32ms/step - accuracy: 0.7704 - loss: 0.4863 - val_accuracy: 0.9155 - val_loss: 0.4414
Epoch 3/10
359/359 ━━━━━━━━━━━━━━━━━━━━ 15s 41ms/step - accuracy: 0.7856 - loss: 0.4591 - val_accuracy: 0.9157 - val_loss: 0.4354
Epoch 4/10
359/359 ━━━━━━━━━━━━━━━━━━━━ 16s 28ms/step - accuracy: 0.8309 - loss: 0.4185 - val_accuracy: 0.7414 - val_loss: 0.5434
Epoch 5/10
359/359 ━━━━━━━━━━━━━━━━━━━━ 12s 32ms/step - accuracy: 0.8647 - loss: 0.3789 - val_accuracy: 0.9173 - val_loss: 0.4226
Epoch 6/10
359/359 ━━━━━━━━━━━━━━━━━━━━ 11s 30ms/step - accuracy: 0.8758 - loss: 0.3512 - val_accuracy: 0.4285 - val_loss: 0.7555
Epoch 7/10
359/359 ━━━━━━━━━━━━━━━━━━━━ 9s 25ms/step - accuracy: 0.8841 - loss: 0.3352 - val_accuracy: 0.9204 - v

### 4. Evaluate and Save

In [35]:
print("\nEvaluating on Test Set (DS2)...")
y_pred_prob = model.predict(X_test, batch_size=128)
y_pred = (y_pred_prob > 0.5).astype(int)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Normal (0)', 'Abnormal (1)']))

os.makedirs('/content/drive/MyDrive/models', exist_ok=True)
model_path = '/content/drive/MyDrive/models/baseline_cnn.keras'
model.save(model_path)
print(f"\nBaseline Model Size: {os.path.getsize(model_path) / 1024:.2f} KB")


Evaluating on Test Set (DS2)...
393/393 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step

Classification Report:
              precision    recall  f1-score   support

  Normal (0)       0.94      0.87      0.90     44653
Abnormal (1)       0.36      0.58      0.44      5609

    accuracy                           0.84     50262
   macro avg       0.65      0.72      0.67     50262
weighted avg       0.88      0.84      0.85     50262


Baseline Model Size: 74.79 KB
